# 36. 하드 AND 와 소프트 점수가 갈리는 지점

입력: 노트북 35 의 `35_baseline_per_query.csv` (3,600행). **재계산하지 않는다.**

노트북 35 는 `P@5` 로 두 검색 방식을 비교했는데 **차이가 0.001 수준이었다.**
지표가 질문에 맞지 않았기 때문이다 — 두 방식의 차이는 *"상위 5개가 좋은가"* 가 아니라
*"애초에 후보가 남는가"* 에서 벌어진다.

이 노트북은 **조건 개수로 층화해** 그 지점을 찾는다.

## 0. 실행 조건과 한계

- **계산하지 않는다.** 노트북 35 가 이미 낸 쿼리별 결과를 층화만 한다
- 조건은 `c3`(사전 + 정규화) 하나만 본다. 조건 1·2 는 노트북 35 에 있다
- **조건 완화를 넣지 않은 상태의 값이다.** `spec.md` §3 ③ 의 5단계 완화를 켜면
  하드 AND 의 실패가 줄어든다. 그 효과는 여기서 재지 않았다
- 층마다 표본 크기가 다르다. **조건 5개 이상은 39건·10건이라 추세만 본다**

In [1]:
import pathlib

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.width", 240)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False

COND = "c3"        # 사전 + 정규화
MAX_BIN = 6        # 조건 개수 상한 (이상은 묶는다)
SMALL_N = 40       # 이 미만이면 '표본 작음' 으로 표시

print("REPORT_ONLY:", REPORT_ONLY, "/ 조건:", COND)

REPORT_ONLY: False / 조건: c3


## 1. 경로 · 쓰기 가드

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"

INPUT_PATH = OUTPUT_DIR / "35_baseline_per_query.csv"
OUTPUT_PATHS = {"stratified": OUTPUT_DIR / "36_hard_vs_soft_stratified.csv"}
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}

if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"노트북 35 의 출력이 없습니다: {INPUT_PATH}")


def write_output(path, writer):
    """OUTPUT_PATHS 경로에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path not in ALLOWED_WRITES:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


q = pd.read_csv(INPUT_PATH)
d = q[q["조건"] == COND].copy()
d["n"] = d["target_n"].clip(upper=MAX_BIN)
print(f"입력 {len(q):,}행 → 조건 {COND} {len(d):,}행 (600문장 × 검색 2)")

입력 3,600행 → 조건 c3 1,200행 (600문장 × 검색 2)


## 2. 왜 집계가 가렸는가

전체 평균으로는 두 방식이 같아 보인다. 층화하면 갈린다.

In [3]:
flat = d.groupby("검색")[["P@5_strict", "검색 불가", "결과 부족"]].mean().round(4)
display(Markdown("**전체 평균 — 구분되지 않는다**"))
display(flat)

**전체 평균 — 구분되지 않는다**

,P@5_strict,검색 불가,결과 부족
검색,,,
hard,0.2281,0.2667,0.275
soft,0.2307,0.2250,0.225


## 3. 조건 개수별 후보 수 — 여기서 갈린다

In [4]:
sizes = d[d["검색"] == "hard"].groupby("n").size().rename("쿼리 수")
cand = d.pivot_table(index="n", columns="검색", values="후보 수", aggfunc="median")
cand["소프트/하드"] = (cand["soft"] / cand["hard"].replace(0, np.nan)).round(1)
table = pd.concat([sizes, cand.round(0)], axis=1)
table["표본"] = ["작음" if v < SMALL_N else "" for v in table["쿼리 수"]]
display(table)

,쿼리 수,hard,soft,소프트/하드,표본
n,,,,,
0,135,0.0,0.0,NaN,
1,81,59969.0,59969.0,1.0,
2,158,10858.0,73843.0,7.0,
3,114,1908.0,86870.0,46.0,
4,63,174.0,94952.0,546.0,
5,39,26.0,111479.0,4288.0,작음
6,10,14.0,122138.0,9047.0,작음


### 읽는 법

**하드 AND 는 조건이 늘수록 후보가 붕괴한다.** 조건 5개면 26개, 6개면 16개가 남는다.
소프트는 반대로 늘어난다 — 조건이 많을수록 하나라도 걸리는 향수가 많아지기 때문이다.

조건 **1개일 때는 둘이 완전히 같다.** 교집합과 합집합이 같아지는 지점이다.

## 4. 5위 동점 — 순위가 나오는가

In [5]:
tie = d.pivot_table(index="n", columns="검색", values="5위 동점", aggfunc="median")
display(pd.concat([sizes, tie.round(0)], axis=1))

,쿼리 수,hard,soft
n,,,
0,135,0.0,0.0
1,81,17736.0,17736.0
2,158,37.0,50.0
3,114,2.0,2.0
4,63,1.0,1.0
5,39,1.0,1.0
6,10,1.0,1.0


조건 **1개면 두 방식 모두 5위 동점이 17,736개**다. 상위 5개가 제비뽑기가 된다.

`DECISIONS.md` N2 가 단독 매핑을 금지한 근거가 이 숫자였고, 여기서 그대로 재현된다.
**중요한 것은 이 실패가 하드/소프트와 무관하다는 점이다.** 조건이 하나뿐인 쿼리는
검색 방식을 바꿔서 구제되지 않는다 — 구조화와 사전이 풀어야 할 문제다.

## 5. 실패율 — 검색 불가와 결과 부족

In [6]:
fail = d.pivot_table(index="n", columns="검색",
                     values=["검색 불가", "결과 부족"], aggfunc="mean").round(3)
display(pd.concat([sizes, fail], axis=1))

h = d[d["검색"] == "hard"]
has_target = h[h["target_n"] > 0]
zero = has_target[has_target["후보 수"] == 0]
s = d[d["검색"] == "soft"]
zero_s = s[(s["target_n"] > 0) & (s["후보 수"] == 0)]
print(f"조건이 있는데 후보가 0개 — 하드 {len(zero)}건 / {len(has_target)}건 "
      f"({len(zero)/max(len(has_target),1):.1%})")
print(f"                        소프트 {len(zero_s)}건")
print("  하드의 조건 개수별 분포:", zero.groupby("target_n").size().to_dict())

,쿼리 수,"(검색 불가, hard)","(검색 불가, soft)","(결과 부족, hard)","(결과 부족, soft)"
n,,,,,
0,135,1.000,1.0,1.000,1.0
1,81,0.000,0.0,0.000,0.0
2,158,0.006,0.0,0.019,0.0
3,114,0.070,0.0,0.079,0.0
4,63,0.079,0.0,0.079,0.0
5,39,0.205,0.0,0.231,0.0
6,10,0.300,0.0,0.400,0.0


조건이 있는데 후보가 0개 — 하드 25건 / 465건 (5.4%)
                        소프트 0건
  하드의 조건 개수별 분포: {2: 1, 3: 8, 4: 5, 5: 8, 6: 2, 9: 1}


**소프트는 조건이 하나라도 있으면 후보가 0이 되지 않는다.** 하드는 5.4% 에서 0이 된다.

`spec.md` §3 ③ 의 조건 완화가 정확히 이 경우를 위해 있다.
**즉 하드 AND 는 완화 장치 없이는 쓸 수 없다.** 소프트는 그 장치가 필요 없다.

## 6. P@5 를 층화하면 — 그래도 크게 안 갈린다

In [7]:
p5 = d.pivot_table(index="n", columns="검색", values="P@5_strict", aggfunc="mean").round(3)
p5["소프트−하드"] = (p5["soft"] - p5["hard"]).round(3)
out = pd.concat([sizes, p5], axis=1)
out["표본"] = ["작음" if v < SMALL_N else "" for v in out["쿼리 수"]]
display(out)
print("표본이 작은 층(조건 5·6개)의 차이는 노이즈와 구별되지 않는다.")

,쿼리 수,hard,soft,소프트−하드,표본
n,,,,,
0,135,0.000,0.000,0.000,
1,81,0.289,0.289,0.000,
2,158,0.260,0.266,0.006,
3,114,0.312,0.318,0.006,
4,63,0.254,0.311,0.057,
5,39,0.429,0.328,-0.101,작음
6,10,0.400,0.440,0.040,작음


표본이 작은 층(조건 5·6개)의 차이는 노이즈와 구별되지 않는다.


**후보 수는 수천 배 차이가 나는데 `P@5` 는 거의 같다.**

후보를 몇 개 남겼는지와 상위 5개가 좋은지는 다른 질문이기 때문이다.
하드가 94,000개를 버려도 그 향수들은 어차피 5위 안에 못 든다.

그래서 **`P@5` 만으로 하드/소프트를 고르면 안 된다.** 고르는 근거는
*"실패가 어디서 나는가"* 쪽이다.

## 7. 갈래별 조건 개수 분포 — 어느 문장이 위험한가

In [8]:
dist = pd.crosstab(h["n"], h["arm"], normalize="columns").round(3)
display(pd.concat([pd.crosstab(h["n"], h["arm"]), dist.add_suffix(" 비율")], axis=1))
print("갈래 A 는 조건 4개 이상이 31.4%, 갈래 B 는 6.0% 다.")
print("하드 AND 의 붕괴는 조건이 많은 쪽에서 일어나므로 갈래 A 가 더 위험하다.")

arm,A,B,A 비율,B 비율
n,,,,
0,35,100,0.117,0.333
1,26,55,0.087,0.183
2,79,79,0.263,0.263
3,66,48,0.220,0.160
4,51,12,0.170,0.040
5,35,4,0.117,0.013
6,8,2,0.027,0.007


갈래 A 는 조건 4개 이상이 31.4%, 갈래 B 는 6.0% 다.
하드 AND 의 붕괴는 조건이 많은 쪽에서 일어나므로 갈래 A 가 더 위험하다.


## 8. 저장

In [9]:
rows = []
for n, g in d.groupby("n"):
    hh, ss = g[g["검색"] == "hard"], g[g["검색"] == "soft"]
    rows.append({
        "조건 개수": int(n), "쿼리 수": len(hh),
        "후보 중앙 하드": hh["후보 수"].median(), "후보 중앙 소프트": ss["후보 수"].median(),
        "5위동점 중앙 하드": hh["5위 동점"].median(),
        "5위동점 중앙 소프트": ss["5위 동점"].median(),
        "검색불가 하드": round(hh["검색 불가"].mean(), 3),
        "검색불가 소프트": round(ss["검색 불가"].mean(), 3),
        "결과부족 하드": round(hh["결과 부족"].mean(), 3),
        "결과부족 소프트": round(ss["결과 부족"].mean(), 3),
        "P@5 하드": round(hh["P@5_strict"].mean(), 3),
        "P@5 소프트": round(ss["P@5_strict"].mean(), 3),
        "표본 작음": len(hh) < SMALL_N,
    })
strat = pd.DataFrame(rows)
display(strat)
write_output(OUTPUT_PATHS["stratified"],
             lambda p: strat.to_csv(p, index=False, encoding="utf-8-sig"))

,조건 개수,쿼리 수,후보 중앙 하드,후보 중앙 소프트,5위동점 중앙 하드,5위동점 중앙 소프트,검색불가 하드,검색불가 소프트,결과부족 하드,결과부족 소프트,P@5 하드,P@5 소프트,표본 작음
0,0,135,0.0,0.0,0.0,0.0,1.000,1.0,1.000,1.0,0.000,0.000,False
1,1,81,59969.0,59969.0,17736.0,17736.0,0.000,0.0,0.000,0.0,0.289,0.289,False
2,2,158,10857.5,73843.0,37.0,50.5,0.006,0.0,0.019,0.0,0.260,0.266,False
3,3,114,1908.0,86870.0,2.0,2.0,0.070,0.0,0.079,0.0,0.312,0.318,False
4,4,63,174.0,94952.0,1.0,1.0,0.079,0.0,0.079,0.0,0.254,0.311,False
5,5,39,26.0,111479.0,1.0,1.0,0.205,0.0,0.231,0.0,0.429,0.328,True
6,6,10,13.5,122137.5,1.0,1.0,0.300,0.0,0.400,0.0,0.400,0.440,True


저장: analysis_outputs\36_hard_vs_soft_stratified.csv


WindowsPath('C:/Users/SSAFY/Desktop/hyanghae/EDA/analysis_outputs/36_hard_vs_soft_stratified.csv')

## 9. 이 노트북이 말하는 것과 말하지 않는 것

**말하는 것**

- 두 방식은 **실패 방향이 반대**다. 하드는 조건이 많을 때, 소프트는 (하드와 똑같이)
  조건이 1개일 때 무너진다
- **소프트가 하드보다 나쁜 구간이 없다.** 검색 불가 0% · 후보 0개 0건
- 하드 AND 는 **조건 완화 장치에 의존한다.** 조건 5개에서 23% 가 결과 부족이다
- 조건 1개 쿼리의 제비뽑기는 **검색 방식으로 못 고친다.** 구조화·사전의 몫이다

**말하지 않는 것**

- **어느 쪽 추천이 더 좋은가.** `P@5` 는 두 방식을 구분하지 못했고, 후보 수는
  품질 지표가 아니다. 사람 판정이나 실사용 데이터가 있어야 답할 수 있다
- **완화를 켰을 때의 하드 AND.** 재지 않았다
- **설문 155건에서도 같은가.** 이 평가셋은 합성이고 조건 개수 분포가 다를 수 있다